In [4]:
def sample_ref_exp(N, rate=8.0, xmax=1.0, rng=None):
    # truncated at 1
    rng = np.random.default_rng() if rng is None else rng
    Z = 1.0 - np.exp(-rate * xmax)
    u = rng.random(N)
    return -(1.0 / rate) * np.log(1.0 - u * Z)

def sample_signal_gauss(N, mu=0.8, sigma=0.02, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    return rng.normal(loc=mu, scale=sigma, size=N)

def make_data_sample_poisson(NR=2000, NS=10, rng=None):
    rng = np.random.default_rng() if rng is None else rng
    N_B = int(rng.poisson(NR))
    N_S = int(rng.poisson(NS))

    xb = sample_ref_exp(N_B, rng=rng)
    xs = sample_signal_gauss(N_S, rng=rng)

    x = np.concatenate([xb, xs])
    rng.shuffle(x)
    return x, xb, xs, N_B, N_S



In [6]:
# --------------------------------
# Parameters
# --------------------------------

seed = 0
rng = np.random.default_rng(seed)

N_R = 200000
NR  = 2000        # expected background under H0

sigma = 0.3 # from estimate_sigma_median

cfg = {
    "sigma": float(sigma),
    "NR": float(NR),
    "M": 3000, #"sqrt",
    "lambda": [1e-10],
    "cpu": False,
    "verbose": 0,
}

Iteration 0 - penalty 1.000000e-10 - sub-iterations 1000000



NPLM statistic t = 30.668386978574134
Size of reference = 200000
Expected background NR = 2000.0
Size of data sample = 1983
Realized N_B, N_S  = 1970 13


In [ ]:
# --------------------------------
# Null toys (H0: data ~ reference)
# --------------------------------

NS = 0          # no signal in data

B = 300         # number of null toys

t_null = np.empty(B)

for b in range(B):
    print(f"\n--- Null toy {b+1}/{B} ---")
    # new reference for each toy
    x_ref_b = sample_ref_exp(N_R, rng=rng)

    # null data: NS = 0
    x_dat_b, _, _, _, _ = make_data_sample_poisson(NR=NR, NS=NS, rng=rng)

    X_b = np.concatenate([x_ref_b, x_dat_b]).reshape(-1, 1).astype(np.float64)
    y_b = np.concatenate([
        np.zeros(len(x_ref_b)),
        np.ones(len(x_dat_b)),
    ]).astype(np.float64)

    nplm = LogFalkonNPLM(cfg)
    t_null[b] = nplm.compute_statistic(X_b, y_b)

np.save("nplm_null_stats.npy", t_null)

In [14]:
# -------------------------
# Alternative toys (H1: data = background + signal)
# -------------------------

NS = 10         # expected signal under H1

B = 100 # number of alt toys (can be smaller than null toys if you just want a quick sanity check)

t_alt  = np.empty(B)

for b in range(B):
    print(f"\n--- Alt toy {b+1}/{B} ---")

    # sample reference
    x_ref_b = sample_ref_exp(NR, rng=rng)

    # alternative data (NS = 10)
    x_dat_b, _, _, N_B_b, N_S_b = make_data_sample_poisson(
        NR=NR,
        NS=NS,
        rng=rng
    )

    X_b = np.concatenate([x_ref_b, x_dat_b]).reshape(-1, 1).astype(np.float64)
    y_b = np.concatenate([
        np.zeros(len(x_ref_b)),
        np.ones(len(x_dat_b)),
    ]).astype(np.float64)

    nplm = LogFalkonNPLM(cfg)
    t_alt[b] = nplm.compute_statistic(X_b, y_b)


--- Alt toy 1/100 ---
Iteration 0 - penalty 1.000000e-10 - sub-iterations 1000000


KeyboardInterrupt: 